In [1]:
import pandas as pd
import sys
from pathlib import Path

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error, r2_score

In [3]:
# Project root (one level above notebooks/)
PROJECT_ROOT = Path.cwd().parents[0]
sys.path.append(str(PROJECT_ROOT))

In [4]:
DATA_PATH = PROJECT_ROOT / "data" / "preprocessed" / "state_level_features.csv"

In [5]:
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

df = df.sort_values(['state_code', 'commodity', 'date']).reset_index(drop=True)

In [6]:
feature_cols = [
    'lag_1', 'lag_2', 'lag_3',
    'lag_6', 'lag_9', 'lag_12',
    'rolling_mean_3', 'rolling_mean_6',
    'rolling_std_3'
]

target_col = 'total_allocated_qty'

In [7]:
def predict_allocation(
    df,
    feature_cols,
    start_date,
    state_code,
    commodity
):
    """
    Single-step prediction (t+1) for allocation.

    Parameters:
    ----------
    df : feature-engineered dataframe
    feature_cols : list of feature columns
    start_date : cutoff date (string or datetime)
    state_code : selected state
    commodity : selected commodity

    Returns:
    -------
    prediction : float
    prediction_date : datetime
    """

    # --------------------------------------------------
    # Prepare Data
    # --------------------------------------------------
    start_date = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)

    df_sc = df[
        (df["state_code"] == state_code) &
        (df["commodity"] == commodity)
    ].copy()

    df_sc = df_sc.sort_values("date").reset_index(drop=True)

    # --------------------------------------------------
    # Train Model (only past data)
    # --------------------------------------------------
    train_df = df_sc[df_sc["date"] <= start_date]

    if train_df.empty:
        raise ValueError("No training data available before selected date.")

    X_train = train_df[feature_cols]
    y_train = train_df["total_allocated_qty"]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)

    # --------------------------------------------------
    # Get Prediction Row
    # --------------------------------------------------
    pred_row_df = df_sc[df_sc["date"] == start_date]

    if pred_row_df.empty:
        raise ValueError(f"No data found for {start_date.date()}")

    pred_row = pred_row_df.iloc[0]

    X_pred = pred_row[feature_cols].values.reshape(1, -1)

    # --------------------------------------------------
    # Predict
    # --------------------------------------------------
    prediction = model.predict(X_pred)[0]

    prediction_date = start_date + pd.DateOffset(months=1)

    return prediction, prediction_date

In [8]:
pred, pred_date = predict_allocation(
    df,
    feature_cols,
    start_date="2021-01-01",
    state_code="KL",
    commodity="rice",
)

print(pred, pred_date)

75426.11682499986 2021-02-28 00:00:00


e:\Learning\Data Science\Python_Projects\pds_foodgrain_forecasting\penv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
